# Section 8b: Pre-Compute Vignette Crops to Disk
One-time script. Reads every image in the split manifests, applies the vignette crop, and saves the cropped version to a new folder. Updates the manifest CSVs to point to the new paths. After this runs, the training pipeline uses pure TF ops (no py_function, no OpenCV at training time).


In [1]:
%run 01_config.ipynb

import os
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
from preprocessing_utils import detect_and_crop_vignette


Creating output structure in: D:\SKIN CANCER/pipeline_output
Set basic random seeds to 42.
No GPU detected. Processing on CPU.


## Setup Output Directory for Cropped Images


In [2]:
CROPPED_ROOT = os.path.join(OUTPUT_ROOT, "cropped_images")
os.makedirs(CROPPED_ROOT, exist_ok=True)

# Create class subdirectories
for cls in CLASS_NAMES:
    os.makedirs(os.path.join(CROPPED_ROOT, cls), exist_ok=True)

print(f"Cropped images will be saved to: {CROPPED_ROOT}")


Cropped images will be saved to: D:\SKIN CANCER/pipeline_output\cropped_images


## Load All Split Manifests


In [3]:
d_splits = os.path.join(OUTPUT_ROOT, "splits")

df_train = pd.read_csv(os.path.join(d_splits, "train_manifest.csv"))
df_val = pd.read_csv(os.path.join(d_splits, "val_manifest.csv"))
df_test = pd.read_csv(os.path.join(d_splits, "test_manifest.csv"))

print(f"Train: {len(df_train)}, Val: {len(df_val)}, Test: {len(df_test)}")
print(f"Total: {len(df_train) + len(df_val) + len(df_test)}")


Train: 14171, Val: 2927, Test: 2957
Total: 20055


## Crop and Save All Images


In [4]:
def crop_and_save_all(df, split_name):
    """
    For each image in the manifest:
    1. Read the original image
    2. Apply vignette crop
    3. Save the cropped image to CROPPED_ROOT/<class>/<filename>
    4. Return the updated paths
    """
    new_paths = []
    crop_stats = {"cropped": 0, "unchanged": 0, "errors": 0}

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Cropping {split_name}"):
        original_path = row["full_path"]
        cls = row["final_authoritative_label"]
        filename = os.path.basename(original_path)

        # Output path
        out_path = os.path.join(CROPPED_ROOT, cls, filename)

        # Skip if already processed (for re-runs)
        if os.path.exists(out_path):
            new_paths.append(out_path)
            continue

        # Read
        img_bgr = cv2.imread(original_path)
        if img_bgr is None:
            print(f"  ERROR reading: {original_path}")
            new_paths.append(original_path)  # keep original path as fallback
            crop_stats["errors"] += 1
            continue

        # Crop
        cropped, did_crop, _, _, _ = detect_and_crop_vignette(img_bgr)

        if did_crop:
            crop_stats["cropped"] += 1
        else:
            crop_stats["unchanged"] += 1

        # Save (cropped or original — detect_and_crop_vignette returns original if no crop)
        cv2.imwrite(out_path, cropped)
        new_paths.append(out_path)

    print(f"  {split_name}: {crop_stats['cropped']} cropped, {crop_stats['unchanged']} unchanged, {crop_stats['errors']} errors")
    return new_paths, crop_stats


print("Processing all images...\n")

train_paths, train_stats = crop_and_save_all(df_train, "train")
val_paths, val_stats = crop_and_save_all(df_val, "val")
test_paths, test_stats = crop_and_save_all(df_test, "test")

print(f"\nDone. All cropped images saved to: {CROPPED_ROOT}")


Processing all images...



Cropping train: 100%|██████████| 14171/14171 [03:02<00:00, 77.82it/s] 


  train: 1084 cropped, 13087 unchanged, 0 errors


Cropping val: 100%|██████████| 2927/2927 [00:36<00:00, 79.36it/s] 


  val: 225 cropped, 2702 unchanged, 0 errors


Cropping test: 100%|██████████| 2957/2957 [01:02<00:00, 47.22it/s] 

  test: 236 cropped, 2721 unchanged, 0 errors

Done. All cropped images saved to: D:\SKIN CANCER/pipeline_output\cropped_images


## Update Manifests with New Paths


In [5]:
# Save original paths for reference, update full_path to cropped versions
df_train["original_path"] = df_train["full_path"]
df_train["full_path"] = train_paths

df_val["original_path"] = df_val["full_path"]
df_val["full_path"] = val_paths

df_test["original_path"] = df_test["full_path"]
df_test["full_path"] = test_paths

# Save updated manifests
d_splits_v2 = os.path.join(OUTPUT_ROOT, "splits")

df_train.to_csv(os.path.join(d_splits_v2, "train_manifest_cropped.csv"), index=False)
df_val.to_csv(os.path.join(d_splits_v2, "val_manifest_cropped.csv"), index=False)
df_test.to_csv(os.path.join(d_splits_v2, "test_manifest_cropped.csv"), index=False)

print("Updated manifests saved:")
print(f"  {d_splits_v2}/train_manifest_cropped.csv")
print(f"  {d_splits_v2}/val_manifest_cropped.csv")
print(f"  {d_splits_v2}/test_manifest_cropped.csv")

# Verify counts
print(f"\nTrain: {len(df_train)} rows")
print(f"Val: {len(df_val)} rows")
print(f"Test: {len(df_test)} rows")

# Verify all new paths exist
for name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    missing = df[~df["full_path"].apply(os.path.exists)]
    if len(missing) > 0:
        print(f"  WARNING: {len(missing)} missing files in {name}!")
    else:
        print(f"  {name}: all files verified on disk.")


Updated manifests saved:
  D:\SKIN CANCER/pipeline_output\splits/train_manifest_cropped.csv
  D:\SKIN CANCER/pipeline_output\splits/val_manifest_cropped.csv
  D:\SKIN CANCER/pipeline_output\splits/test_manifest_cropped.csv

Train: 14171 rows
Val: 2927 rows
Test: 2957 rows
  train: all files verified on disk.
  val: all files verified on disk.
  test: all files verified on disk.


## Summary


In [6]:
total_cropped = train_stats["cropped"] + val_stats["cropped"] + test_stats["cropped"]
total_unchanged = train_stats["unchanged"] + val_stats["unchanged"] + test_stats["unchanged"]
total_images = total_cropped + total_unchanged

print("=== SECTION 8b SUMMARY ===")
print(f"Total images processed: {total_images}")
print(f"Vignette crop applied: {total_cropped} ({100*total_cropped/total_images:.1f}%)")
print(f"Left unchanged: {total_unchanged}")
print(f"Cropped images saved to: {CROPPED_ROOT}")
print(f"")
print("=== DOWNSTREAM INSTRUCTIONS ===")
print("In notebooks 10, 11, 12: load the *_cropped.csv manifests instead of the originals.")
print("Use the PURE TF preprocessing function (no py_function, no OpenCV):")
print("  preprocess_image() with tf.io.read_file -> tf.image.decode_jpeg -> resize -> center_crop -> normalize")
print("The vignette crop is already baked into the saved images.")
print("")
print("Original manifests are untouched. Original images are untouched.")


=== SECTION 8b SUMMARY ===
Total images processed: 20055
Vignette crop applied: 1545 (7.7%)
Left unchanged: 18510
Cropped images saved to: D:\SKIN CANCER/pipeline_output\cropped_images

=== DOWNSTREAM INSTRUCTIONS ===
In notebooks 10, 11, 12: load the *_cropped.csv manifests instead of the originals.
Use the PURE TF preprocessing function (no py_function, no OpenCV):
  preprocess_image() with tf.io.read_file -> tf.image.decode_jpeg -> resize -> center_crop -> normalize
The vignette crop is already baked into the saved images.

Original manifests are untouched. Original images are untouched.
